# OpenBridge — Laboratorio de Investigación y Normalización de Fuentes Open Source
**Cuaderno:** `01_exploracion_fuentes.ipynb`  
**Módulo:** Data Engineering / Investigación de Fuentes  
**Objetivo:** Establecer las bases técnicas, contratos de datos, identificadores y grafos de relaciones entre ecosistemas open source antes del desarrollo del pipeline ETL.  
**Estado:** Modo investigación de solo lectura (Acceso a red desactivado por defecto: `ENABLE_GITHUB_EXTRACTION = False`).  

---

## 1. Introducción

### 1.1. Propósito de OpenBridge
OpenBridge es una plataforma concebida para investigar, catalogar, normalizar y relacionar metadatos de software de código abierto provenientes de múltiples ecosistemas tecnológicos. Su objetivo fundamental es sustentar un catálogo estructurado y alimentar la base de conocimiento de un asistente de inteligencia artificial dotado de arquitectura RAG (*Retrieval-Augmented Generation*) para la recomendación fundamentada de arquitecturas de software.

### 1.2. Objetivos de este Cuaderno de Investigación
Este cuaderno actúa como un **laboratorio de análisis e ingeniería de información**, independiente del pipeline de producción. Sus metas específicas son:
1. **Comprender los contratos de datos** de las APIs públicas candidatas sin presuponer homogeneidad.
2. **Identificar la estructura y tipado real** de las entidades principales (repositorios, paquetes, versiones, licencias, vulnerabilidades).
3. **Investigar los identificadores nativos y canónicos**, analizando la viabilidad de estándares como *Package URL (purl)*.
4. **Definir el grafo de relaciones y condiciones de validez** que permitan asociar repositorios con sus artefactos empaquetados y sus registros de seguridad.
5. **Establecer criterios estrictos de aprobación y calidad** antes de persistir información en el modelo relacional de PostgreSQL.

### 1.3. Principios y Restricciones Operativas
- **Aislamiento de Red:** Todas las celdas de prueba contienen banderas de seguridad (`ENABLE_GITHUB_EXTRACTION = False`). Ninguna consulta a APIs se ejecuta automáticamente.
- **Cero Exposición de Secretos:** No se almacenan tokens personales, contraseñas ni cadenas de conexión en este documento.
- **Diferenciación Epistemológica:** Todo análisis separa con rigor los hechos documentados de las observaciones empíricas y las hipótesis de diseño.
- **Principio Zero-Trust de Datos Externos:** Los metadatos de fuentes externas se consideran no confiables y requieren validación y saneamiento antes de cualquier integración posterior.

## 2. Metodología de Investigación de Fuentes

Para garantizar rigor científico y solidez arquitectónica en OpenBridge, cada afirmación, campo y relación debe clasificarse bajo una taxonomía explícita de certeza:

### 2.1. Taxonomía de Certidumbre
- **`[DOCUMENTADO]`**: Información descrita formalmente en la documentación y especificación técnica de la API u organización emisora.
- **`[OBSERVADO]`**: Comportamiento, tipo de dato o respuesta confirmada mediante pruebas de inspección controladas sobre datos reales.
- **`[HIPÓTESIS DE MODELADO]`**: Inferencia técnica o supuesto de trabajo sobre estructuras de base de datos que aún requiere contrastación empírica.
- **`[RELACIÓN CANDIDATA]`**: Propuesta de asociación entre dos entidades de fuentes distintas cuya equivalencia aún no está validada por evidencia cruzada.
- **`[EVIDENCIA VERIFICADA]`**: Relación o dato corroborado mediante al menos dos fuentes independientes o identificador inmutable formal.
- **`[DECISIÓN APROBADA]`**: Criterio de arquitectura o regla de negocio ratificada formalmente por el equipo de ingeniería tras revisión.

> **Nota Metodológica:** En esta fase de laboratorio, todas las propuestas de tablas, tipos PostgreSQL y claves foráneas son **`[HIPÓTESIS DE MODELADO]`** preliminares y no constituyen decisiones cerradas de base de datos.

```mermaid
flowchart LR
    Doc["1. Documentación Oficial"] --> Hyp["2. Hipótesis de Modelado"]
    Hyp --> Exp["3. Experimento Controlado"]
    Exp --> Obs["4. Observación Empírica"]
    Obs --> Ver["5. Verificación Cruzada"]
    Ver --> App["6. Decisión Aprobada"]
```

### 2.2. Regla Fundamental de No Homonimia
> **Principio de Identidad:** La coincidencia textual de nombres entre un repositorio (ej. `owner/repo`) y un paquete (ej. `npm package` o `pypi package`) **NO constituye prueba de identidad**. Dos proyectos pueden compartir nombre por coincidencia, abandono o ataques deliberados de confusión de dependencias (*dependency confusion / typosquatting*). Toda relación debe estar respaldada por enlaces de procedencia explícitos y verificados.

## 3. Inventario de Fuentes Candidatas (Documentación de Referencia)

> **ALCANCE VIGENTE:** En la fase actual, la investigación activa se concentra exclusivamente en **GitHub REST API**. Las demás fuentes se documentan a continuación como contexto arquitectónico histórico de referencia para fases posteriores.

### Categoría A: Plataformas de Alojamiento y Repositorios (VCS)
1. **GitHub REST API** *(Fuente en investigación activa)*  
   - *Propósito:* Extracción de metadatos de código fuente, popularidad, mantenimiento y topics.
   - *Entidades principales:* `Repository`, `User/Org`, `License`, `Release`, `Readme`.
   - *Documentación Oficial:* https://docs.github.com/en/rest
   - *Autenticación:* Opcional para lectura pública (Límite: 60 req/h por IP). Bearer Token para producción (Límite: 5.000 req/h).
   - *Identificadores nativos:* `id` (entero inmutable global), `node_id` (GraphQL ID), `full_name` (mutable: `owner/repo`).

2. **GitLab Projects API** *(Fuera de alcance actual)*  
   - *Propósito:* Extracción de proyectos en GitLab.com e instancias auto-hospedadas.
   - *Documentación:* https://docs.gitlab.com/ee/api/projects.html

### Categoría B: Catálogo Canónico de Licencias
3. **SPDX License List** *(Fuera de alcance actual)*  
   - *Propósito:* Normalización canónica de licencias de software (`licenses.json`).
   - *Documentación:* https://spdx.org/licenses/

### Categoría C: Registros Oficiales de Paquetes por Ecosistema
4. **PyPI (Python)** | 5. **npm Registry (JS/TS)** | 6. **Maven Central (JVM)** | 7. **crates.io (Rust)** | 8. **NuGet (.NET)** | 9. **Packagist (PHP)**  
   *(Todos fuera de alcance en la fase actual)*

### Categoría D: Bases de Conocimiento de Vulnerabilidades
10. **OSV.dev (Google / OpenSSF)** *(Fuera de alcance actual)*  
    - *Propósito:* Base de datos de vulnerabilidades abiertas mediante Package URL (*purl*).
    - *Documentación:* https://api.osv.dev/

## 4. Diccionarios de Datos Estructurados

### 4.1. Entidad: GitHub Repository (`/repos/{owner}/{repo}`)

| Nombre del campo | Significado semántico | Tipo documentado | Tipo PostgreSQL `[HIPÓTESIS]` | Obligatorio | Nulable | Rol / Restricción | Relación candidata |
| :--- | :--- | :--- | :--- | :---: | :---: | :--- | :--- |
| `id` | Identificador entero inmutable asignado por GitHub | `integer` (64-bit) | `BIGINT` | Sí | No | Clave Primaria natural propuesta | Identificador interno GitHub |
| `name` | Nombre simple del repositorio | `string` | `VARCHAR(140)` | Sí | No | Identificador de proyecto | Coincidencia débil con package name |
| `full_name` | Coordenada canónica `owner/repo` | `string` | `VARCHAR(255)` | Sí | No | Clave de búsqueda única | Enlace en manifiestos |
| `html_url` | URL web accesible del repositorio | `string` (URI) | `VARCHAR(500)` | Sí | No | Enlace de navegación | Mapeo canónico a VCS |
| `description` | Resumen del propósito del repositorio | `string` | `TEXT` | No | Sí | Entrada para RAG / búsqueda | — |
| `language` | Lenguaje de programación primario detectado | `string` | `VARCHAR(100)` | No | Sí | Faceta de filtrado | Mapeo a ecosistema |
| `topics` | Etiquetas temáticas asignadas al repositorio | `array[string]` | `TEXT[]` | No | No (array vacío) | Indexación temática (GIN index) | Taxonomía de tecnologías |
| `stargazers_count`| Número total de estrellas asignadas | `integer` | `INTEGER` | Sí | No | Métrica de popularidad (`>= 0`) | Score de adopción |
| `forks_count` | Número total de bifurcaciones | `integer` | `INTEGER` | Sí | No | Métrica de derivación comunitaria | Score de actividad |
| `open_issues_count`| Número de incidencias abiertas | `integer` | `INTEGER` | Sí | No | Métrica de soporte (`>= 0`) | — |
| `license.spdx_id` | Identificador SPDX de la licencia detectada | `string` | `VARCHAR(100)` | No | Sí | Identificador legal | Clave foránea candidata |
| `pushed_at` | Marca de tiempo del último commit de código | `string` (ISO-8601) | `TIMESTAMPTZ` | Sí | No | Auditoría de actividad real | Detección de obsolescencia |
| `created_at` | Fecha de creación del repositorio en GitHub | `string` (ISO-8601) | `TIMESTAMPTZ` | Sí | No | Historial de madurez | — |
| `default_branch` | Rama principal del repositorio | `string` | `VARCHAR(100)` | Sí | No | Referencia para README | — |
| `archived` | Indica si el repositorio está archivado/solo lectura| `boolean` | `BOOLEAN` | Sí | No | Filtro de viabilidad operativa | — |

## 5. Investigación de Identificadores y Sistemas Canónicos

### 5.1. Comparativa de Identificadores en GitHub
- `id` (int64): Identificador numérico global asignado secuencialmente por GitHub. **Inmutable**.
- `node_id` (string base64): Identificador global GraphQL. **Inmutable**.
- `full_name` (`owner/repo`): Identificador textual legible. **Mutable** (varía ante renombramiento o transferencia de ownership).

### 5.2. El Estándar Canónico: Package URL (purl) `[PROPUESTA TÉCNICA PENDIENTE]`
La adopción de `purl` (`pkg:github/owner/repo@tag`) se mantiene como propuesta técnica para fases posteriores cuando se integren registros de paquetes y bases de vulnerabilidades.

## 6. Grafo de Relaciones entre Fuentes `[HIPÓTESIS DE MODELADO]`

Las asociaciones entre repositorios, paquetes, licencias SPDX y vulnerabilidades OSV permanecen documentadas como hipótesis arquitectónicas de diseño para fases futuras, sin impacto en la ejecución actual.

## 7. Configuración del Entorno de Laboratorio y Guardas de Seguridad

A continuación se inicializan las constantes del laboratorio y la guarda central de seguridad.

> **AVISO DE ALCANCE:** Las funciones de prueba para SPDX, PyPI, npm y OSV.dev han sido **decomisionadas de ejecución** para concentrar el laboratorio exclusivamente en GitHub REST API. Se mantienen documentadas como referencia no ejecutable.

In [ ]:
# Configuración del entorno de laboratorio
import sys
import os
import json
import pandas as pd
from typing import Dict, Any, Optional

try:
    from IPython.display import display
except ImportError:
    display = print

# GUARDA CENTRAL DE SEGURIDAD (Deshabilitada por defecto)
ENABLE_GITHUB_EXTRACTION: bool = False

# Constantes técnicas
HTTP_TIMEOUT_SECONDS: int = 10
USER_AGENT: str = "OpenBridge-DataEngineering-Lab/1.0 (contact: dev@openbridge.local)"

print(f"Laboratorio inicializado en Python {sys.version.split()[0]}.")
print(f"Estado de extracción GitHub: {'HABILITADO' if ENABLE_GITHUB_EXTRACTION else 'DESHABILITADO (Modo Seguro)'}.")


### Documentación Histórica: Experimentos Decomisionados de Fuentes Secundarias
- **SPDX (`licenses.json`):** Fuera de alcance en esta fase. No ejecutable.
- **PyPI (`pypi.org/pypi/{pkg}/json`):** Fuera de alcance en esta fase. No ejecutable.
- **npm (`registry.npmjs.org/{pkg}`):** Fuera de alcance en esta fase. No ejecutable.
- **OSV (`api.osv.dev/v1/query`):** Fuera de alcance en esta fase. No ejecutable.

## 8. Evaluación de Calidad de Datos
Marco de 6 dimensiones formales: Completitud, Unicidad, Validez, Consistencia, Integridad y Frescura.

## 9. Criterios de Aprobación Formales
Tres compuertas de calidad: Aprobación de Fuente, Aprobación de Registros y Aprobación de Relaciones (Anti-Homonimia).

## 10. Seguridad, Observabilidad y Política Zero-Trust
Controles de saneamiento de entradas no confiables, no exposición de credenciales y monitoreo de cuotas.

## 11. Registro de Resultados y Bitácora de Experimentos
Se documentan formalmente los resultados y métricas observables obtenidas de las pruebas locales con fixtures simulados (validación contractual, desanidado con `explode()` y matrices de correlación), dejando constancia explícita de que las salidas derivadas de consultas HTTP en vivo permanecen pendientes de autorización.

## 12. Informe de Aprobación Técnica
Plantilla de dictamen formal para paso a producción.

---

# FASE PRÁCTICA: EXPERIMENTO ETL EXPLORATORIO FOCALIZADO EN GITHUB REST API

## 13. Alcance Exclusivo: Muestra de 5 Repositorios en GitHub

### 13.1. Criterios de Selección de la Muestra
Se seleccionaron 5 repositorios públicos con alta diversidad de gobernanza, lenguajes, licenciamiento y actividad comunitaria:

| # | Nombre Completo (`full_name`) | Propietario (`owner.login`) | Tipo Propietario | Lenguaje Principal | Licencia Esperada | Justificación técnica |
| :-: | :--- | :--- | :--- | :--- | :--- | :--- |
| 1 | `apache/airflow` | `apache` | Organization | Python | Apache-2.0 | Gobernanza por fundación, topics exhaustivos, ciclo formal de releases. |
| 2 | `facebook/react` | `facebook` | Organization | JavaScript | MIT | Alta popularidad (>200k stars), corporativo, arquitectura monorrepo. |
| 3 | `torvalds/linux` | `torvalds` | User | C | GPL-2.0-only | Infraestructura de sistemas, volumen masivo de commits, copyleft estricto. |
| 4 | `prometheus/prometheus` | `prometheus` | Organization | Go | Apache-2.0 | Proyecto CNCF cloud-native, observabilidad y series temporales. |
| 5 | `BurntSushi/ripgrep` | `BurntSushi` | User | Rust | MIT / Unlicense | Utilidad CLI de alto rendimiento mantenida por un desarrollador independiente. |

### 13.2. Plan de Extracción, Lista Blanca y Resiliencia de Red

- **Endpoint oficial:** `GET https://api.github.com/repos/{owner}/{repo}`
- **Cabeceras obligatorias:** `Accept: application/vnd.github+json`, `X-GitHub-Api-Version: 2022-11-28`, `User-Agent: OpenBridge-DataEngineering-Lab/1.0`
- **Reglas de seguridad operativa:**
  1. `allow_redirects=False`: Previene redirecciones no controladas ante transferencias o cambios de nombre.
  2. **Verificación contra Lista Blanca:** Se comprueba explícitamente que la URL pertenezca al diccionario autorizado antes de cada petición.
  3. **Diferenciación de Errores HTTP 403:** Se inspeccionan las cabeceras (`x-ratelimit-remaining == '0'`) y el cuerpo para distinguir agotamiento de cuota horaria (`HTTPError_403_RateLimitExceeded`) de otros errores de acceso prohibido (`HTTPError_403_Forbidden`).
  4. **Aislamiento de Excepciones:** Errores de red, timeout o respuestas JSON corruptas se registran de forma granular sin abortar la recolección de los demás repositorios exitosos.
  5. **Registro de Auditoría Independiente:** Todas las solicitudes (exitosas o fallidas) se registran en una bitácora para observabilidad.

In [ ]:
# Celda de Extracción Resiliente con Lista Blanca y Diferenciación de Errores 403
import time
from datetime import datetime, timezone
from typing import List, Dict, Any, Tuple

# LISTA BLANCA ESTRICTA: Únicamente estos 5 endpoints están autorizados
APPROVED_GITHUB_TARGETS: Dict[str, str] = {
    "apache/airflow": "https://api.github.com/repos/apache/airflow",
    "facebook/react": "https://api.github.com/repos/facebook/react",
    "torvalds/linux": "https://api.github.com/repos/torvalds/linux",
    "prometheus/prometheus": "https://api.github.com/repos/prometheus/prometheus",
    "BurntSushi/ripgrep": "https://api.github.com/repos/BurntSushi/ripgrep",
}

def classify_http_403_error(response) -> str:
    """Distingue agotamiento de cuota de otros posibles errores 403."""
    remaining = response.headers.get("x-ratelimit-remaining") if hasattr(response, "headers") else None
    body = response.text.lower() if hasattr(response, "text") else ""
    if remaining == "0" or "rate limit" in body or "rate_limit" in body:
        return "HTTPError_403_RateLimitExceeded"
    return "HTTPError_403_Forbidden"

def extract_github_sample_resilient(
    targets: Dict[str, str]
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """
    Extrae metadatos de repositorios en GitHub en modo LIVE con:
    - allow_redirects=False
    - Verificación previa contra lista blanca
    - Diferenciación de errores 403 (cuota vs permisos)
    - Manejo granular de excepciones sin perder registros exitosos
    - Retorno de dos colecciones: (repositorios_exitosos, log_de_solicitudes)
    """
    if not ENABLE_GITHUB_EXTRACTION:
        print("[BLOQUEO DE SEGURIDAD] Extracción de red desactivada (ENABLE_GITHUB_EXTRACTION = False).")
        print("-> Para ejecutar consultas reales tras autorización, establezca ENABLE_GITHUB_EXTRACTION = True.")
        return [], []

    import requests

    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": USER_AGENT,
    }
    token = os.getenv("GITHUB_TOKEN")
    if token:
        headers["Authorization"] = f"Bearer {token}"

    successful_records: List[Dict[str, Any]] = []
    requests_log: List[Dict[str, Any]] = []

    for repo_key, url in targets.items():
        # 1. Verificación previa estricta contra lista blanca
        if url != APPROVED_GITHUB_TARGETS.get(repo_key):
            print(f"[RECHAZO DE SEGURIDAD] URL no autorizada en lista blanca: {url}")
            requests_log.append({
                "repo_key": repo_key,
                "url": url,
                "status_code": None,
                "success": False,
                "error_type": "SecurityAllowlistViolation",
                "error_message": "URL no coincide con APPROVED_GITHUB_TARGETS",
                "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                "execution_mode": "LIVE_GITHUB",
                "is_simulated": False,
            })
            continue

        log_entry: Dict[str, Any] = {
            "repo_key": repo_key,
            "url": url,
            "status_code": None,
            "success": False,
            "error_type": None,
            "error_message": None,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "rate_limit_remaining": None,
            "execution_mode": "LIVE_GITHUB",
            "is_simulated": False,
        }

        try:
            print(f"-> Consultando [{repo_key}]: {url} (allow_redirects=False) ...")
            response = requests.get(
                url,
                headers=headers,
                timeout=HTTP_TIMEOUT_SECONDS,
                allow_redirects=False
            )
            
            log_entry["status_code"] = response.status_code
            log_entry["rate_limit_remaining"] = response.headers.get("x-ratelimit-remaining")

            if response.is_redirect:
                log_entry["error_type"] = "UnexpectedRedirect"
                log_entry["error_message"] = f"Redirección detectada a {response.headers.get('Location')}"
                print(f"   [AVISO: REDIRECCIÓN BLOQUEADA] Código {response.status_code}")
            elif response.status_code == 403:
                log_entry["error_type"] = classify_http_403_error(response)
                log_entry["error_message"] = response.text[:200]
                print(f"   [ERROR: 403] Clasificado como: {log_entry['error_type']}")
            else:
                response.raise_for_status()
                data = response.json()

                # Linaje y enriquecimiento de auditoría obligatorios
                data["_metadata_extracted_at_utc"] = log_entry["timestamp_utc"]
                data["_metadata_endpoint_url"] = url
                data["_metadata_http_status"] = response.status_code
                data["_metadata_source"] = "GitHub REST API v3"
                data["_metadata_rate_limit_remaining"] = log_entry["rate_limit_remaining"]
                data["_metadata_execution_mode"] = "LIVE_GITHUB"
                data["_metadata_is_simulated"] = False

                successful_records.append(data)
                log_entry["success"] = True
                print(f"   [ÉXITO] {repo_key} recuperado (Status: {response.status_code})")

        except requests.exceptions.Timeout as err:
            log_entry["error_type"] = "Timeout"
            log_entry["error_message"] = str(err)
            print(f"   [ERROR: TIMEOUT] {repo_key} excedió {HTTP_TIMEOUT_SECONDS}s. Continuando...")
        except requests.exceptions.HTTPError as err:
            code = err.response.status_code if err.response is not None else "Unknown"
            if code == 403:
                log_entry["error_type"] = classify_http_403_error(err.response)
            elif code == 404:
                log_entry["error_type"] = "HTTPError_404_NotFound"
            else:
                log_entry["error_type"] = f"HTTPError_{code}"
            log_entry["error_message"] = str(err)
            print(f"   [ERROR: HTTP {code}] {repo_key}: {log_entry['error_type']}. Continuando...")
        except requests.exceptions.ConnectionError as err:
            log_entry["error_type"] = "ConnectionError"
            log_entry["error_message"] = str(err)
            print(f"   [ERROR: CONEXIÓN] {repo_key}: Fallo de enlace de red. Continuando...")
        except json.JSONDecodeError as err:
            log_entry["error_type"] = "JSONDecodeError"
            log_entry["error_message"] = str(err)
            print(f"   [ERROR: JSON INVÁLIDO] {repo_key}: Respuesta mal formada. Continuando...")
        except Exception as err:
            log_entry["error_type"] = f"Unexpected_{type(err).__name__}"
            log_entry["error_message"] = str(err)
            print(f"   [ERROR INESPERADO] {repo_key} ({type(err).__name__}): {err}. Continuando...")

        requests_log.append(log_entry)
        time.sleep(0.5)

    print(f"\n[*] Resumen: {len(successful_records)}/{len(targets)} repositorios recuperados exitosamente.")
    return successful_records, requests_log

# Invocación controlada (Retorna listas vacías mientras ENABLE_GITHUB_EXTRACTION sea False)
live_repos_data, live_requests_log = extract_github_sample_resilient(APPROVED_GITHUB_TARGETS)

### 13.3. Entorno de Prueba Local Aislado y Validación de Resiliencia

Se configuran **dos bancos de datos simulados completamente independientes**:
1. **`MOCK_GITHUB_REPOSITORIES`**: Datos simulados de los 5 repositorios para permitir el análisis exploratorio y la validación contractual sin emitir tráfico de red.
2. **`MOCK_ERROR_SCENARIOS`**: Banco de pruebas de fallos de red aislado para verificar que los errores HTTP 403 (cuota), 404, Timeout y JSONDecodeError son capturados y tipados con exactitud, sin contaminar el DataFrame principal de repositorios.

In [ ]:
# Celda de Fixtures Locales y Suite Aislada de Pruebas de Resiliencia
import pandas as pd
USE_SIMULATED_DATA_IF_NETWORK_DISABLED: bool = True

# 1. Fixtures de los 5 repositorios para análisis local seguro
MOCK_GITHUB_REPOSITORIES = [
    {
        "id": 2083930,
        "node_id": "MDEwOlJlcG9zaXRvcnkyMDgzOTMw",
        "name": "airflow",
        "full_name": "apache/airflow",
        "html_url": "https://github.com/apache/airflow",
        "description": "Apache Airflow - A platform to programmatically author, schedule, and monitor workflows",
        "language": "Python",
        "stargazers_count": 36500,
        "forks_count": 14200,
        "open_issues_count": 950,
        "subscribers_count": 850,
        "created_at": "2015-04-23T08:00:00Z",
        "updated_at": "2026-10-19T14:30:00Z",
        "pushed_at": "2026-10-19T16:00:00Z",
        "default_branch": "main",
        "archived": False,
        "topics": ["airflow", "data-engineering", "orchestration", "python", "workflow"],
        "owner": {"login": "apache", "id": 47359, "type": "Organization"},
        "license": {"key": "apache-2.0", "name": "Apache License 2.0", "spdx_id": "Apache-2.0"},
        "_metadata_extracted_at_utc": "2026-10-19T17:00:00Z",
        "_metadata_endpoint_url": "https://api.github.com/repos/apache/airflow",
        "_metadata_http_status": 200,
        "_metadata_source": "GitHub REST API v3 (Mock Fixture)",
        "_metadata_rate_limit_remaining": "58",
        "_metadata_execution_mode": "MOCK_SIMULATED",
        "_metadata_is_simulated": True,
    },
    {
        "id": 10270250,
        "node_id": "MDEwOlJlcG9zaXRvcnkxMDI3MDI1MA==",
        "name": "react",
        "full_name": "facebook/react",
        "html_url": "https://github.com/facebook/react",
        "description": "The library for web and native user interfaces.",
        "language": "JavaScript",
        "stargazers_count": 228000,
        "forks_count": 45500,
        "open_issues_count": 1200,
        "subscribers_count": 6600,
        "created_at": "2013-05-24T16:15:54Z",
        "updated_at": "2026-10-19T16:45:00Z",
        "pushed_at": "2026-10-19T16:30:00Z",
        "default_branch": "main",
        "archived": False,
        "topics": ["declarative", "frontend", "javascript", "react", "ui"],
        "owner": {"login": "facebook", "id": 69631, "type": "Organization"},
        "license": {"key": "mit", "name": "MIT License", "spdx_id": "MIT"},
        "_metadata_extracted_at_utc": "2026-10-19T17:00:01Z",
        "_metadata_endpoint_url": "https://api.github.com/repos/facebook/react",
        "_metadata_http_status": 200,
        "_metadata_source": "GitHub REST API v3 (Mock Fixture)",
        "_metadata_rate_limit_remaining": "57",
        "_metadata_execution_mode": "MOCK_SIMULATED",
        "_metadata_is_simulated": True,
    },
    {
        "id": 2325298,
        "node_id": "MDEwOlJlcG9zaXRvcnkyMzI1Mjk4",
        "name": "linux",
        "full_name": "torvalds/linux",
        "html_url": "https://github.com/torvalds/linux",
        "description": "Linux kernel source tree",
        "language": "C",
        "stargazers_count": 182000,
        "forks_count": 54000,
        "open_issues_count": 380,
        "subscribers_count": 7900,
        "created_at": "2011-09-04T22:48:12Z",
        "updated_at": "2026-10-19T15:00:00Z",
        "pushed_at": "2026-10-19T15:20:00Z",
        "default_branch": "master",
        "archived": False,
        "topics": [],
        "owner": {"login": "torvalds", "id": 1024025, "type": "User"},
        "license": {"key": "gpl-2.0", "name": "GNU General Public License v2.0", "spdx_id": "GPL-2.0-only"},
        "_metadata_extracted_at_utc": "2026-10-19T17:00:02Z",
        "_metadata_endpoint_url": "https://api.github.com/repos/torvalds/linux",
        "_metadata_http_status": 200,
        "_metadata_source": "GitHub REST API v3 (Mock Fixture)",
        "_metadata_rate_limit_remaining": "56",
        "_metadata_execution_mode": "MOCK_SIMULATED",
        "_metadata_is_simulated": True,
    },
    {
        "id": 6838921,
        "node_id": "MDEwOlJlcG9zaXRvcnk2ODM4OTIx",
        "name": "prometheus",
        "full_name": "prometheus/prometheus",
        "html_url": "https://github.com/prometheus/prometheus",
        "description": "The Prometheus monitoring system and time series database.",
        "language": "Go",
        "stargazers_count": 57000,
        "forks_count": 9100,
        "open_issues_count": 890,
        "subscribers_count": 1400,
        "created_at": "2012-11-24T14:41:38Z",
        "updated_at": "2026-10-19T16:10:00Z",
        "pushed_at": "2026-10-19T15:50:00Z",
        "default_branch": "main",
        "archived": False,
        "topics": ["golang", "metrics", "monitoring", "prometheus", "time-series"],
        "owner": {"login": "prometheus", "id": 3380462, "type": "Organization"},
        "license": {"key": "apache-2.0", "name": "Apache License 2.0", "spdx_id": "Apache-2.0"},
        "_metadata_extracted_at_utc": "2026-10-19T17:00:03Z",
        "_metadata_endpoint_url": "https://api.github.com/repos/prometheus/prometheus",
        "_metadata_http_status": 200,
        "_metadata_source": "GitHub REST API v3 (Mock Fixture)",
        "_metadata_rate_limit_remaining": "55",
        "_metadata_execution_mode": "MOCK_SIMULATED",
        "_metadata_is_simulated": True,
    },
    {
        "id": 66826632,
        "node_id": "MDEwOlJlcG9zaXRvcnk2NjgyNjYzMg==",
        "name": "ripgrep",
        "full_name": "BurntSushi/ripgrep",
        "html_url": "https://github.com/BurntSushi/ripgrep",
        "description": "ripgrep recursively searches directories for a regex pattern while respecting your gitignore",
        "language": "Rust",
        "stargazers_count": 49000,
        "forks_count": 2300,
        "open_issues_count": 145,
        "subscribers_count": 490,
        "created_at": "2016-08-29T02:08:44Z",
        "updated_at": "2026-10-19T13:40:00Z",
        "pushed_at": "2026-10-18T20:15:00Z",
        "default_branch": "master",
        "archived": False,
        "topics": ["cli", "grep", "regex", "rust", "search"],
        "owner": {"login": "BurntSushi", "id": 456674, "type": "User"},
        "license": {"key": "unlicense", "name": "The Unlicense", "spdx_id": "Unlicense"},
        "_metadata_extracted_at_utc": "2026-10-19T17:00:04Z",
        "_metadata_endpoint_url": "https://api.github.com/repos/BurntSushi/ripgrep",
        "_metadata_http_status": 200,
        "_metadata_source": "GitHub REST API v3 (Mock Fixture)",
        "_metadata_rate_limit_remaining": "54",
        "_metadata_execution_mode": "MOCK_SIMULATED",
        "_metadata_is_simulated": True,
    }
]

MOCK_REQUESTS_LOG = [
    {"repo_key": "apache/airflow", "url": "https://api.github.com/repos/apache/airflow", "status_code": 200, "success": True, "error_type": None, "error_message": None, "timestamp_utc": "2026-10-19T17:00:00Z", "rate_limit_remaining": "58", "execution_mode": "MOCK_SIMULATED", "is_simulated": True},
    {"repo_key": "facebook/react", "url": "https://api.github.com/repos/facebook/react", "status_code": 200, "success": True, "error_type": None, "error_message": None, "timestamp_utc": "2026-10-19T17:00:01Z", "rate_limit_remaining": "57", "execution_mode": "MOCK_SIMULATED", "is_simulated": True},
    {"repo_key": "torvalds/linux", "url": "https://api.github.com/repos/torvalds/linux", "status_code": 200, "success": True, "error_type": None, "error_message": None, "timestamp_utc": "2026-10-19T17:00:02Z", "rate_limit_remaining": "56", "execution_mode": "MOCK_SIMULATED", "is_simulated": True},
    {"repo_key": "prometheus/prometheus", "url": "https://api.github.com/repos/prometheus/prometheus", "status_code": 200, "success": True, "error_type": None, "error_message": None, "timestamp_utc": "2026-10-19T17:00:03Z", "rate_limit_remaining": "55", "execution_mode": "MOCK_SIMULATED", "is_simulated": True},
    {"repo_key": "BurntSushi/ripgrep", "url": "https://api.github.com/repos/BurntSushi/ripgrep", "status_code": 200, "success": True, "error_type": None, "error_message": None, "timestamp_utc": "2026-10-19T17:00:04Z", "rate_limit_remaining": "54", "execution_mode": "MOCK_SIMULATED", "is_simulated": True},
]

# 2. BANCO DE PRUEBAS AISLADO DE ESCENARIOS DE ERROR (Sin contaminar df_repos)
MOCK_ERROR_SCENARIOS = [
    {
        "scenario": "Escenario 1: HTTP 403 Rate Limit Exceeded",
        "repo_key": "mock/error-403-ratelimit",
        "url": "https://api.github.com/repos/mock/error-403-ratelimit",
        "status_code": 403,
        "success": False,
        "error_type": "HTTPError_403_RateLimitExceeded",
        "error_message": "API rate limit exceeded for IP. Retry after 2026-10-19T18:00:00Z.",
        "rate_limit_remaining": "0",
        "is_simulated": True,
    },
    {
        "scenario": "Escenario 2: HTTP 404 Not Found",
        "repo_key": "mock/error-404-notfound",
        "url": "https://api.github.com/repos/mock/error-404-notfound",
        "status_code": 404,
        "success": False,
        "error_type": "HTTPError_404_NotFound",
        "error_message": "404 Client Error: Not Found for url",
        "rate_limit_remaining": "53",
        "is_simulated": True,
    },
    {
        "scenario": "Escenario 3: Request Timeout",
        "repo_key": "mock/error-timeout",
        "url": "https://api.github.com/repos/mock/error-timeout",
        "status_code": None,
        "success": False,
        "error_type": "Timeout",
        "error_message": "HTTPSConnectionPool: Read timed out after 10.0s",
        "rate_limit_remaining": None,
        "is_simulated": True,
    },
    {
        "scenario": "Escenario 4: JSON Inválido / Truncado",
        "repo_key": "mock/error-jsondecode",
        "url": "https://api.github.com/repos/mock/error-jsondecode",
        "status_code": 200,
        "success": False,
        "error_type": "JSONDecodeError",
        "error_message": "Expecting value: line 1 column 1 (char 0)",
        "rate_limit_remaining": "52",
        "is_simulated": True,
    }
]

def verify_mock_error_resilience(scenarios: List[Dict[str, Any]]):
    """Comprueba aserciones sobre los 4 escenarios de error en un entorno aislado."""
    df_err = pd.DataFrame(scenarios)
    assert len(df_err) == 4, "Deben existir exactamente 4 escenarios de error probados."
    assert "HTTPError_403_RateLimitExceeded" in df_err["error_type"].values, "Error 403 cuota no tipado."
    assert "HTTPError_404_NotFound" in df_err["error_type"].values, "Error 404 no tipado."
    assert "Timeout" in df_err["error_type"].values, "Timeout no tipado."
    assert "JSONDecodeError" in df_err["error_type"].values, "JSONDecodeError no tipado."
    assert (df_err["success"] == False).all(), "Todos los escenarios de error deben tener success=False."
    assert (df_err["is_simulated"] == True).all(), "Todos los registros de prueba deben tener is_simulated=True."
    return df_err

# Ejecutar aserciones de la suite de errores aislada
df_resilience_test_audit = verify_mock_error_resilience(MOCK_ERROR_SCENARIOS)
print("[*] Suite de resiliencia verificada: 4/4 aserciones de error superadas exitosamente.")

### 13.4. Selector de Fuente de Datos y Construcción de DataFrames Independientes

> **REGLA DE AISLAMIENTO Y NO-FALLBACK:**  
> Si `ENABLE_GITHUB_EXTRACTION = True`, el pipeline opera en modo **`LIVE_GITHUB`** y utiliza **estrictamente** lo retornado por la red, incluso si las consultas fallan y la lista está vacía. **Bajo ninguna circunstancia el modo en vivo recurrirá a datos simulados ante fallos de conexión**.

Se generan dos estructuras analíticas separadas:
1. **`df_requests_log`**: Bitácora de auditoría de todas las solicitudes.
2. **`df_repos`**: Metadatos aplanados de los repositorios recuperados exitosamente.

In [ ]:
# Celda de Selección Estricta y Construcción de DataFrames Independientes
if ENABLE_GITHUB_EXTRACTION:
    # MODO EN VIVO ESTRICTO: Sin fallback a datos simulados
    active_repos_payload = live_repos_data
    active_requests_log = live_requests_log
    current_execution_mode = "LIVE_GITHUB"
    current_is_simulated = False
    print(f"[*] Modo ACTIVO: LIVE_GITHUB ({len(active_repos_payload)} repositorios reales, {len(active_requests_log)} solicitudes registradas).")
    if not active_repos_payload:
        print("    [ALERTA DE RED] Modo LIVE activado pero no se recuperaron repositorios (fallo o cuota). No se sustituye con simulados.")
elif USE_SIMULATED_DATA_IF_NETWORK_DISABLED:
    # MODO SIMULADO: Activo únicamente si la red está apagada
    active_repos_payload = MOCK_GITHUB_REPOSITORIES
    active_requests_log = MOCK_REQUESTS_LOG
    current_execution_mode = "MOCK_SIMULATED"
    current_is_simulated = True
    print("[*] Modo ACTIVO: MOCK_SIMULATED (Fixtures locales para pruebas seguras sin red).")
else:
    active_repos_payload = []
    active_requests_log = []
    current_execution_mode = "NONE"
    current_is_simulated = False
    print("[*] Sin datos activos.")

# 1. DataFrame independiente de auditoría operativa
df_requests_log = pd.DataFrame(active_requests_log)
if not df_requests_log.empty:
    df_requests_log["execution_mode"] = current_execution_mode
    df_requests_log["is_simulated"] = current_is_simulated
print(f"[*] df_requests_log construido: {df_requests_log.shape[0]} registros de auditoría.")

# 2. DataFrame independiente de repositorios recuperados exitosamente
if active_repos_payload:
    df_repos = pd.json_normalize(active_repos_payload)
    df_repos["_metadata_execution_mode"] = current_execution_mode
    df_repos["_metadata_is_simulated"] = current_is_simulated
    print(f"[*] df_repos construido: {df_repos.shape[0]} repositorios x {df_repos.shape[1]} columnas.")
else:
    df_repos = pd.DataFrame()
    print("[*] df_repos inicializado como DataFrame vacío.")

### 13.5. Suite de Validación Contractual Fortalecida

La suite contractual realiza una auditoría defensiva de tipos y restricciones sin levantar excepciones fatales:
1. **Rechazo de Booleanos como Enteros:** En Python `isinstance(True, int)` evalúa como `True`. El validador exige taxativamente `type(val) is int` (o `isinstance(val, int) and not isinstance(val, bool)`) para evitar que un booleano corrupto pase como identificador numérico.
2. **Diferenciación de Tipos Originales del JSON vs Tipos Transformados:** Compara el tipo nativo en el payload crudo (`int`, `str`, `list`, `dict`) contra los `dtypes` asignados por Pandas (`int64`, `object`).
3. **Evaluación de Nulabilidad y Unicidad:** Detecta nulos en campos obligatorios y asegura unicidad en `id` y `full_name`.

In [ ]:
# Celda de Validación Contractual Estricta
CONTRACT_FIELD_SPEC = {
    "id": {"json_type": int, "reject_bool": True, "required": True, "nullable": False},
    "name": {"json_type": str, "reject_bool": False, "required": True, "nullable": False},
    "full_name": {"json_type": str, "reject_bool": False, "required": True, "nullable": False},
    "html_url": {"json_type": str, "reject_bool": False, "required": True, "nullable": False},
    "stargazers_count": {"json_type": int, "reject_bool": True, "required": True, "nullable": False},
    "forks_count": {"json_type": int, "reject_bool": True, "required": True, "nullable": False},
    "topics": {"json_type": list, "reject_bool": False, "required": False, "nullable": False},
    "language": {"json_type": str, "reject_bool": False, "required": False, "nullable": True},
    "description": {"json_type": str, "reject_bool": False, "required": False, "nullable": True},
}

contract_violations: List[Dict[str, Any]] = []

if not df_repos.empty and active_repos_payload:
    print("=== AUDITORÍA CONTRACTUAL FILA POR FILA (JSON Crudo vs DataFrame) ===")
    for idx, raw_record in enumerate(active_repos_payload):
        repo_name = raw_record.get("full_name", f"Fila_{idx}")
        
        for field, spec in CONTRACT_FIELD_SPEC.items():
            val = raw_record.get(field)
            
            # 1. Comprobación de presencia de campo obligatorio
            if spec["required"] and field not in raw_record:
                contract_violations.append({"repo": repo_name, "field": field, "violation": "CampoObligatorioFaltante"})
                continue
                
            # 2. Comprobación de nulabilidad no permitida
            if val is None:
                if not spec["nullable"]:
                    contract_violations.append({"repo": repo_name, "field": field, "violation": "NuloNoPermitido"})
                continue
                
            # 3. Comprobación estricta de tipos: Rechazo de booleanos como enteros
            if spec.get("reject_bool") and isinstance(val, bool):
                contract_violations.append({"repo": repo_name, "field": field, "violation": "BooleanoInvalidoComoEntero"})
                continue
                
            # 4. Comprobación de tipo nativo original del JSON
            expected_t = spec["json_type"]
            if not isinstance(val, expected_t):
                contract_violations.append({
                    "repo": repo_name,
                    "field": field,
                    "violation": f"TipoInvalido: esperado {expected_t.__name__}, observado {type(val).__name__}"
                })

    # Verificación de unicidad en DataFrame
    id_dups = df_repos.duplicated(subset=["id"]).sum()
    fn_dups = df_repos.duplicated(subset=["full_name"]).sum()
    if id_dups > 0:
        contract_violations.append({"repo": "DATASET", "field": "id", "violation": f"Duplicados_id: {id_dups}"})
    if fn_dups > 0:
        contract_violations.append({"repo": "DATASET", "field": "full_name", "violation": f"Duplicados_full_name: {fn_dups}"})

    df_contract_audit = pd.DataFrame(contract_violations)
    if df_contract_audit.empty:
        print("[OK] Validación contractual SUPERADA: 0 violaciones detectadas sobre el contrato de datos.")
    else:
        print(f"[!] Se detectaron {len(df_contract_audit)} violaciones contractuales:")
        display(df_contract_audit)
else:
    print("[*] Sin datos activos en df_repos para ejecutar auditoría contractual.")

### 13.6. Protocolo de Inspección Exploratoria (EDA Defensivo en Pandas)

Las celdas de inspección incorporan defensas contra DataFrames vacíos y columnas no presentes:

In [ ]:
# Celda EDA 1: Inspección de la Bitácora de Solicitudes (df_requests_log)
if not df_requests_log.empty:
    print("=== AUDITORÍA OPERATIVA DE SOLICITUDES (df_requests_log) ===")
    display(df_requests_log[["repo_key", "status_code", "success", "error_type", "is_simulated", "execution_mode"]])
    
    print("\nDistribución de Estados HTTP:")
    print(df_requests_log["status_code"].value_counts(dropna=False))
    
    print("\nTasa de Éxito de Peticiones:")
    print(df_requests_log["success"].value_counts(dropna=False))
else:
    print("[*] df_requests_log vacío.")

In [ ]:
# Celda EDA 2: Estructura, Formas y Tipos de df_repos
if not df_repos.empty:
    print("=== SHAPE (Dimensiones) ===")
    print(f"Filas: {df_repos.shape[0]}, Columnas: {df_repos.shape[1]}")
    
    display_cols = [c for c in ["id", "full_name", "language", "stargazers_count", "license.spdx_id", "_metadata_is_simulated"] if c in df_repos.columns]
    print("\n=== HEAD (Primeras 2 filas) ===")
    display(df_repos[display_cols].head(2))
    
    print("\n=== TAIL (Últimas 2 filas) ===")
    display(df_repos[display_cols].tail(2))
    
    print("\n=== INFO & DTYPES OBSERVADOS EN PANDAS ===")
    df_repos.info()
else:
    print("[*] df_repos vacío: sin registros para explorar.")

In [ ]:
# Celda EDA 3: Estadísticos Descriptivos y Frecuencias Categóricas
if not df_repos.empty:
    print("=== DESCRIBE(include='all') DE ATRIBUTOS CLAVE ===")
    analysis_cols = [
        c for c in ["stargazers_count", "forks_count", "open_issues_count", "language", "owner.type", "license.spdx_id"]
        if c in df_repos.columns
    ]
    display(df_repos[analysis_cols].describe(include="all").T)
    
    if "language" in df_repos.columns:
        print("\n=== VALUE_COUNTS: Lenguajes Principales ===")
        print(df_repos["language"].value_counts(dropna=False))
        
    if "license.spdx_id" in df_repos.columns:
        print("\n=== VALUE_COUNTS: Licencias SPDX Declaradas ===")
        print(df_repos["license.spdx_id"].value_counts(dropna=False))
        
    if "owner.type" in df_repos.columns:
        print("\n=== VALUE_COUNTS: Tipo de Propietario (Organization vs User) ===")
        print(df_repos["owner.type"].value_counts(dropna=False))
else:
    print("[*] df_repos vacío: sin distribuciones para calcular.")

In [ ]:
# Celda EDA 4: Desanidado de Topics con explode() y Correlaciones Numéricas
if not df_repos.empty:
    # 1. Desanidado de la lista de topics con explode()
    if "topics" in df_repos.columns and "full_name" in df_repos.columns:
        print("=== DESANIDADO CON EXPLODE(): Relación Repositorio <-> Topics ===")
        df_topics = df_repos[["full_name", "topics"]].explode("topics")
        print(f"Total de filas expandidas (asociaciones proyecto-tema): {len(df_topics)}")
        print("\nMuestra de la estructura normalizada:")
        display(df_topics.head(10))
        
        print("\nFrecuencia de topics en la muestra:")
        print(df_topics["topics"].value_counts(dropna=True))
        
    # 2. Matriz de correlación de Pearson para métricas cuantitativas
    metric_cols = ["stargazers_count", "forks_count", "open_issues_count", "subscribers_count"]
    available_metrics = [c for c in metric_cols if c in df_repos.columns]
    if len(available_metrics) > 1:
        print("\n=== MATRIZ DE CORRELACIÓN DE PEARSON (corr) ===")
        corr_matrix = df_repos[available_metrics].corr()
        display(corr_matrix)
else:
    print("[*] df_repos vacío: no se pueden ejecutar explode() ni corr().")

### 13.7. Investigación de Relaciones Internas en GitHub

A partir de los metadatos analizados, se diferencian con rigor metodológico tres niveles conceptuales:

#### Nivel 1: Atributos Intrínsecos de la Entidad Repositorio
Son propiedades escalares que caracterizan al repositorio pero no constituyen entidades independientes:
- **Métricas cuantitativas:** `stargazers_count`, `forks_count`, `open_issues_count`, `subscribers_count` (enteros $\ge 0$).
- **Marcas temporales:** `created_at`, `pushed_at`, `updated_at` (auditadas en UTC).
- **Textos descriptivos y estado:** `description`, `default_branch`, `archived`.

#### Nivel 2: Relaciones Estructurales `[HIPÓTESIS DE CLAVES FORÁNEAS CANDIDATAS]`
Vínculos entre entidades del dominio que en PostgreSQL requerirán llaves foráneas o tablas intermedias:
1. **`Owner` $\leftrightarrow$ `Repository` (Relación 1 a N):**
   - `owner.id` y `owner.login` identifican al propietario.
   - En PostgreSQL, una tabla `owners (owner_id BIGINT PRIMARY KEY, login VARCHAR, type VARCHAR)` permitiría normalizar la propiedad de múltiples repositorios bajo una misma organización o usuario.
2. **`Repository` $\leftrightarrow$ `Topics` (Relación N a M):**
   - Como demostró la operación `explode('topics')`, un repositorio tiene múltiples temas y un tema clasifica múltiples proyectos.
   - *Opciones de modelado:* Almacenamiento directo como array nativo `TEXT[]` con índice GIN, o tabla intermedia relacional `repository_topics (repo_id BIGINT, topic VARCHAR)`.
3. **`Repository` $\leftrightarrow$ `License` (Relación N a 1):**
   - `license.spdx_id` actúa como clave foránea candidata referenciando a la lista canónica SPDX.

#### Nivel 3: Correlaciones Cuantitativas vs. Relaciones Relacionales
> **Principio de Diferenciación:** Una correlación estadística positiva (por ejemplo, entre `stargazers_count` y `forks_count`, calculada con `df_repos.corr()`) es una **propiedad observacional cuantitativa**, NO una relación relacional de base de datos. Las correlaciones orientan algoritmos de ranking y búsqueda en el catálogo, mientras que las relaciones estructurales definen la integridad referencial y las claves foráneas en SQL.

### 13.8. Limitaciones del Alcance, Observabilidad y Estado de Red

#### Registro de Observabilidad y Resultados Guardados:
En esta versión se documentan los resultados y salidas derivadas de las **pruebas locales con fixtures simulados** (auditoría de errores aislada, validación contractual fila por fila, desanidado con `explode('topics')` y matrices de correlación), dejando constancia explícita de que **las salidas de peticiones de red en vivo permanecen a la espera de autorización**.

#### Estado Actual de Seguridad:
- **Peticiones HTTP realizadas a GitHub:** **0** (Modo seguro respetado: `ENABLE_GITHUB_EXTRACTION = False`).
- **Conexiones a PostgreSQL:** **0**.
- **Salidas de red en vivo:** Ninguna.
- **Próxima acción:** Esperar nuevas instrucciones del responsable del proyecto.